# Population-level trends for Client Outcome Measures

## Changes in average (over clients) scores ATOM Assessment Survey questions.

## Steps:

1. Extract the data - from the database or from a pre-prepared parquet file
2. Processing: if not pulling from pre-cleaned data: Clean & Transform (incl. categorize) the data
    - Clean data - remove rows with missing data : PDCSubstanceOrGambling
    - Transform data - expand PDC, determine Program, categorize fields, drop Notes/Comments fields, rename PartitionKey to SLK.
    - Limit the data to the period of interest - i.e. only clients who have completed at least one survey during the period of interest.
    - Limit by only clients who have completed the survey at least three times (min-stage: 3)
5. Calculate the average score for each client for each stage and for each of the questions of interest.


In [1]:

# Step 0: Importing the libraries
from utils.df_xtrct_prep import extract_prep_data

from statsutil.funcs import get_all_results
from utils.io import write_results_to_files, create_results_folder
from graphing import get_chart_for_qna_list

In [2]:
# Global variables
extract_start_date = 20200101
extract_end_date = 20240101

period_folder = f"{extract_start_date}_{extract_end_date}_1"

active_clients_start_date ='2022-07-01' 
active_clients_end_date = '2023-06-30'

results_folder = "./data/out"


# MIN_NUM_ATOMS_PER_CLIENT = 3
# MIN_NUM_COL_VALUES = 3

### Step 1 & 2: Extract & Process

#### Extract the data - from the database or from a pre-prepared parquet file

1. *Processed data*:
  - if processed-parquet file is not present, *get the raw data* and process it and cache it into the parquet file.
  - if yes, load the data from the parquet file.
  
2. If *Raw data* doesn't exist in the data/in/ folder as a parquet file:
  - load it from the database (Azure)
  - otherwise from the parquet file.
 
 (cache=True => try to load from a parquet file, if not present, load from the database and cache it into a parquet file)

In [3]:
# Extract & Process
processed_df = extract_prep_data(extract_start_date, extract_end_date
                                 , active_clients_start_date
                                 , active_clients_end_date
                                 , period_folder, min_atoms_per_client =3)

In [4]:
# len(processed_df)
processed_df.info()

<class 'pandas.core.frame.DataFrame'>
Index: 2478 entries, 7083 to 2250
Columns: 186 entries, SLK to PDCGoals
dtypes: category(24), datetime64[ns](1), float64(15), object(146)
memory usage: 3.1+ MB


In [5]:
# processed_df['DiversityList'].dropna().empty
# empty columns:
blank_cols_indf = [col for col in processed_df.columns 
                   if processed_df[col].dropna().empty ]
processed_df.drop(columns=blank_cols_indf, inplace=True)
# for col in processed_df.columns:
#     if processed_df[col].dropna().empty:
#         print("Dropping column " , col)
#         processed_df.drop(columns=col, inplace=True)
# 18 columns dropped
# Relationship
# ManagingImpact
# LegalIssuesDueToOthersUse
# HowCloseToManagingImpactOfOthersUse
# ReferredFrom
# DiversityList
# HowLongSmoked
# ConfidentQuittingReducing
# HowManyTimesQuitSmoking
# HowManagingQuitting
# HowMuchDoYouKnow
# InterventionType
# QuitlineReferral
# CountryofBirth
# ClientId
# SubstanceUse
# K10CompletedBy
# HONOSAssessment

### Step 3 : Calculate the average score for each client for each stage and for each of the questions of interest.

In [28]:
# Chronologically Rank the Assessments for each client
# df_q = chrono_rank_within_clientgroup(processed_df)  # adds 'survey_rank' column
# g = col_df.groupby('SLK')
# col_df.loc[:,'survey_rank'] = g['AssessmentDate'].rank(method='min')

In [6]:
chosen_surveys = [1, 3 ,6]

In [7]:
from filters import get_filters, apply_filters, get_outfilename_for_filters

In [8]:
def do_all(df, chosen_surveys, orig_filter):

  outfile_name = get_outfilename_for_filters(orig_filter)
  
  filters = get_filters(orig_filter , exclude_fields=['FunderName'])
  new_df = apply_filters(df, filters)

  all_results = get_all_results(new_df, chosen_surveys, filters)

  write_results_to_files(all_results, f"{results_folder}/{period_folder}/{outfile_name}.csv")

  return all_results


In [10]:
# orig_filter2 = {   'Program':['EUROPATH'] }

# create period folders to prep for results .csv files to be dropped in
create_results_folder(f"{results_folder}/{period_folder}")

orig_filter1 = {'FunderName': 'NSW Ministry of Health'}
MoH_results = do_all(processed_df, chosen_surveys, orig_filter1)

# orig_filter1 = {'FunderName': 'Coordinaire'}
# Coordinaire_results = do_all(processed_df, chosen_surveys, orig_filter1)




# orig_filter1 = {'FunderName': 'Murrumbidgee PHN'}
# MPHN_results = do_all(processed_df, chosen_surveys, orig_filter1)

# orig_filter1 = {'FunderName': 'ACT Health'}
# ACTHealth_results = do_all(processed_df, chosen_surveys, orig_filter1)

Wellbeing measures
NRecords For Col(Past4WkPhysicalHealth): 166)#, Total:520, 2021-07-21 00:00:00, 2023-06-29 00:00:00
NRecords For Col(Past4WkMentalHealth): 166)#, Total:520, 2021-07-21 00:00:00, 2023-06-29 00:00:00
NRecords For Col(Past4WkQualityOfLifeScore): 166)#, Total:520, 2021-07-21 00:00:00, 2023-06-29 00:00:00
Substance Use
NRecords For Col(PDCHowMuchPerOccasion): 180)#, Total:520, 2020-04-18 00:00:00, 2023-06-29 00:00:00
NRecords For Col(PDCDaysInLast28): 198)#, Total:520, 2020-04-18 00:00:00, 2023-06-29 00:00:00
Problems in Life Domains
NRecords For Col(Past4WkDailyLivingImpacted): 166)#, Total:520, 2021-07-21 00:00:00, 2023-06-29 00:00:00
NRecords For Col(Past4WkHowOftenPhysicalHealthCausedProblems): 166)#, Total:520, 2021-07-21 00:00:00, 2023-06-29 00:00:00
NRecords For Col(Past4WkHowOftenMentalHealthCausedProblems): 166)#, Total:520, 2021-07-21 00:00:00, 2023-06-29 00:00:00
NRecords For Col(Past4WkUseLedToProblemsWithFamilyFriend): 166)#, Total:520, 2021-07-21 00:00:00, 2

In [10]:


chart , points = get_chart_for_qna_list(question_list, answers_df, title)

chart + points

alt.LayerChart(...)

#### Write results to CSV

In [18]:
from datetime import datetime
title_for_file = title.replace(" ", "_")
results_filepath = f"{results_folder}{period_folder}_{title_for_file}.csv"
# write_df_to_csv(answers_df, f"{results_folder}{fname}_{title_for_file}.csv")
#f"./data/out/results_{fname}.csv"
answers_df['ResultsTimestamp'] = datetime.now().replace(microsecond=0)
answers_df.to_csv(results_filepath, index=False, mode='a', header=True)


In [8]:

# chosen_surveys = [1, 3 ,6] 
# answer_list = get_nmeans_for_questions( question_list, processed_df, chosen_surveys)

NRecords For Col(Past4WkHowOftenPhysicalHealthCausedProblems): 931, Total:2434, 2020-01-07 00:00:00, 2023-06-29 00:00:00
Past4WkHowOftenPhysicalHealthCausedProblems,1,125,1.38
Past4WkHowOftenPhysicalHealthCausedProblems,3,125,1.31
Past4WkHowOftenPhysicalHealthCausedProblems,6,125,1.43
NRecords For Col(Past4WkHowOftenMentalHealthCausedProblems): 931, Total:2434, 2020-01-07 00:00:00, 2023-06-29 00:00:00
Past4WkHowOftenMentalHealthCausedProblems,1,125,2.04
Past4WkHowOftenMentalHealthCausedProblems,3,125,1.66
Past4WkHowOftenMentalHealthCausedProblems,6,125,1.72
NRecords For Col(Past4WkUseLedToProblemsWithFamilyFriend): 931, Total:2434, 2020-01-07 00:00:00, 2023-06-29 00:00:00
Past4WkUseLedToProblemsWithFamilyFriend,1,125,0.74
Past4WkUseLedToProblemsWithFamilyFriend,3,125,0.61
Past4WkUseLedToProblemsWithFamilyFriend,6,125,0.44
NRecords For Col(Past4WkDifficultyFindingHousing): 915, Total:2434, 2020-01-07 00:00:00, 2023-06-29 00:00:00
Past4WkDifficultyFindingHousing,1,123,0.24
Past4WkDifficu

In [9]:

title = "Problems in Life Domains"
#'Changes in average scores for "Past 4 weeks: Use let to problems in various Life domains" '
chart , points = get_chart_for_qna_list(question_list, answer_list, chosen_surveys, title)

chart + points

NameError: name 'get_chart_for_qna_list' is not defined

In [25]:
# col_df1[col_df1['survey_rank'] == 1].Past4WkPhysicalHealth.count() #.value_counts(dropna=False)
# len(col_df1[col_df1['survey_rank'] == 1].SLK.unique() )
# len(df_q[df_q['survey_rank'] == 1].SLK.unique() )


528

In [26]:
# col_df1[col_df1['survey_rank'] == 6].Past4WkPhysicalHealth.count()

# len(col_df1[col_df1['survey_rank'] == 6].SLK.unique() )
# len(df_q[df_q['survey_rank'] == 6].SLK.unique() )



138

In [12]:
## client_groups_forcol = col_df.groupby('SLK')
# from graphing import get_chart_for_means

# question_list = [question]
# assessment_tags= chosen_surveys
# means = averages

# # contribs = [first_assess_contribs,fourth_assess_contribs, seventh_assess_contribs ]
# chart = get_chart_for_means(question_list, assessment_tags, means, nth_assessment_contribs)
# chart

alt.LayerChart(...)